# 05 Uncertainty experiment (Experiment C)

**Question.** Do predicted variances track error, and do split-conformal intervals reach their nominal coverage on an independent partition?

**Type.** Run/inspection ([spec](../docs/experiments/c-uncertainty-aware.md)). Calibration is fitted on the calibration partition only.

In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import dataclasses
import json
import logging

import matplotlib.pyplot as plt
import torch
from IPython.display import display

from ct_iqa.config.loader import load_study_config
from ct_iqa.experiment import report
from ct_iqa.experiment.run import run_experiment
from ct_iqa.experiment.runs import read_predictions, select_run

CONFIG = "configs/experiments/c_uncertainty_aware.yaml"
cfg = load_study_config(CONFIG)
print(cfg.run.experiment_id, "|", cfg.model.name, "|", cfg.loss.name, "| bn_eps =", cfg.model.bn_eps,
      "| epochs =", cfg.train.epochs, "| batch =", cfg.train.batch_size, "| seed =", cfg.train.seed)

import math

from ct_iqa.evaluation.intervals import to_score_scale


## Run (explicit, expensive)
Set `RUN_TRAINING = True` only with an approved compute budget; this trains for the configured number of epochs and writes a new immutable run directory.

In [ ]:
# Training is OFF by default. Set RUN_TRAINING = True only with an approved compute budget: it trains for
# the configured number of epochs and writes a NEW immutable run directory. Progress is logged one line per epoch below the cell.
RUN_TRAINING = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID = None      # name of a completed run to inspect; required once several completed runs exist

if RUN_TRAINING:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
    cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, device=DEVICE))
    run_dir = run_experiment(cfg, config_path=CONFIG)
else:
    run_dir = select_run(cfg.run.experiment_id, RUN_ID)
print("device:", DEVICE, "| run directory:", run_dir if run_dir else "NO COMPLETED RUN FOUND (status PENDING)")


## Point-prediction results

In [ ]:
if run_dir:
    history = json.loads((run_dir / "history.json").read_text())
    metrics = json.loads((run_dir / "metrics_validation.json").read_text())
    print(f"Validation metrics at the selected epoch {history['best_epoch']} (policy: {history['selection_policy']}; "
          f"raw 0-4 scale; selection-biased, not a generalization estimate)")
    display(report.metrics_frame(metrics))
    display(report.history_frame(history).round(4))
    report.plot_history(history, cfg.run.experiment_id)
    plt.show()


## Split-conformal intervals
Fitted on the calibration partition, evaluated on the validation partition. Nominal coverage is not a clinical guarantee; exchangeability is unverified (image-level split, no patient identifiers); with 100 calibration images coverage is only controlled to roughly +/-0.03.

In [ ]:
if run_dir:
    cal = json.loads((run_dir / "calibration.json").read_text())
    tables = report.calibration_frames(cal)
    display(tables["summary"]); display(tables["intervals"].round(4)); display(tables["terciles"].round(4))

    p = read_predictions(run_dir)
    _, sigma = to_score_scale(p["prediction_normalized"], p["log_var_normalized"])
    q = math.inf if cal["quantile_q"] == "inf" else float(cal["quantile_q"])
    report.plot_intervals(p["label_0_4"], p["prediction_0_4"], sigma, q, cal["target_coverage"], tables["terciles"],
                          "Validation intervals")
    plt.show()
